In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/H-IDS")
API_DIR = ROOT / "api"

assert (API_DIR / "main.py").exists(), "api/main.py not found!"

print("Project root:", ROOT)
print("API file:", API_DIR / "main.py")

Mounted at /content/drive
Project root: /content/drive/MyDrive/H-IDS
API file: /content/drive/MyDrive/H-IDS/api/main.py


In [ ]:
%pip -q install fastapi uvicorn httpx

In [ ]:
import sys

if str(API_DIR) not in sys.path:
    sys.path.insert(0, str(API_DIR))

from fastapi.testclient import TestClient
import main

client = TestClient(main.app)

response = client.get("/health")
print("Health status:", response.status_code)
print("Response:", response.json())

response = client.get("/features")
print("Feature endpoint:", response.status_code)
print("Number of features:", len(response.json()["features"]))

/usr/lib/python3.13/pickle.py:1754: UserWarning: [04:22:13] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:439: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [04:22:13] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [04:22:13] WARNING: /__w/xgboost/xgboost/src/context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)


Health status: 200
Response: {'status': 'ok', 'binary_model_loaded': True, 'multiclass_model_loaded': True, 'feature_count': 78, 'class_count': 15}
Feature endpoint: 200
Number of features: 78


/usr/lib/python3.13/pickle.py:1754: UserWarning: [04:22:15] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:439: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [04:22:15] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [04:22:15] WARNING: /__w/xgboost/xgboost/src/context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)


In [ ]:

import pandas as pd

val_path = ROOT / "data" / "splits" / "validation.parquet"
val_df = pd.read_parquet(val_path)

row = val_df.iloc[0]

# Ground-truth label from the dataset
actual_label = str(row["label"])

# Send only the 78 model features to the API
payload = {
    "features": {
        feature: float(row[feature])
        for feature in main.FEATURES
    }
}

response = client.post("/predict", json=payload)

print("HTTP status:", response.status_code)
print("Actual label:", actual_label)

if response.status_code == 200:
    result = response.json()
    print("API response:", result)

    predicted_label = result.get("predicted_label")

    print("\nPrediction:", predicted_label)
    print("Correct prediction:", predicted_label == actual_label)
else:
    print("API error:", response.text)


HTTP status: 200
Actual label: BENIGN
API response: {'binary_decision': 'BENIGN', 'attack_probability': 2.9668152023987204e-07, 'predicted_label': 'BENIGN', 'multiclass_confidence': None, 'threshold': 0.5}

Prediction: BENIGN
Correct prediction: True


In [ ]:
print("Validation columns:")
print(val_df.columns.tolist())

print("\nFirst row:")
display(val_df.head(1))

Validation columns:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Cou

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,label_raw,label,source_file,source_row
0,55254.0,3.0,3.0,0.0,43.0,0.0,31.0,6.0,14.333333,14.433757,...,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN,BENIGN,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,13


In [ ]:

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Sample 100 validation records
sample_df = val_df.sample(
    n=min(100, len(val_df)),
    random_state=42
)

actual_labels = []
predicted_labels = []
errors = []

for idx, row in sample_df.iterrows():
    payload = {
        "features": {
            feature: float(row[feature])
            for feature in main.FEATURES
        }
    }

    try:
        response = client.post("/predict", json=payload)

        if response.status_code != 200:
            errors.append({
                "index": idx,
                "status": response.status_code,
                "error": response.text
            })
            continue

        result = response.json()

        actual_labels.append(str(row["label"]))
        predicted_labels.append(result["predicted_label"])

    except Exception as exc:
        errors.append({
            "index": idx,
            "error": str(exc)
        })

print("Requested records:", len(sample_df))
print("Successful predictions:", len(predicted_labels))
print("Failed requests:", len(errors))

if predicted_labels:
    print(
        "\nAPI accuracy on successfully processed sample:",
        round(accuracy_score(actual_labels, predicted_labels), 6)
    )

    print("\nClassification report:")
    print(classification_report(
        actual_labels,
        predicted_labels,
        zero_division=0
    ))

    print("\nConfusion matrix:")
    labels = sorted(set(actual_labels) | set(predicted_labels))
    print("Label order:", labels)
    print(confusion_matrix(
        actual_labels,
        predicted_labels,
        labels=labels
    ))

if errors:
    print("\nFirst errors:")
    print(errors[:5])


Requested records: 100
Successful predictions: 100
Failed requests: 0

API accuracy on successfully processed sample: 1.0

Classification report:
                        precision    recall  f1-score   support

                BENIGN       1.00      1.00      1.00        74
                  DDoS       1.00      1.00      1.00         5
              DoS Hulk       1.00      1.00      1.00        11
         DoS slowloris       1.00      1.00      1.00         1
              PortScan       1.00      1.00      1.00         8
Web Attack Brute Force       1.00      1.00      1.00         1

              accuracy                           1.00       100
             macro avg       1.00      1.00      1.00       100
          weighted avg       1.00      1.00      1.00       100


Confusion matrix:
Label order: ['BENIGN', 'DDoS', 'DoS Hulk', 'DoS slowloris', 'PortScan', 'Web Attack Brute Force']
[[74  0  0  0  0  0]
 [ 0  5  0  0  0  0]
 [ 0  0 11  0  0  0]
 [ 0  0  0  1  0  0]
 [ 0  0  

In [ ]:
from pathlib import Path

api_file = ROOT / "api" / "main.py"

print(api_file.read_text())


from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field


PROJECT_ROOT = Path(__file__).resolve().parents[1]

BINARY_PATH = (
    PROJECT_ROOT / "models" / "binary" / "xgboost_binary_bundle.joblib"
)
MULTICLASS_PATH = (
    PROJECT_ROOT / "models" / "multiclass" /
    "xgboost_multiclass_bundle.joblib"
)

for path in [BINARY_PATH, MULTICLASS_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"Model bundle not found: {path}")

binary_bundle = joblib.load(BINARY_PATH)
multiclass_bundle = joblib.load(MULTICLASS_PATH)

binary_model = binary_bundle["model"]
binary_imputer = binary_bundle["imputer"]
binary_features = list(binary_bundle["feature_columns"])
binary_mapping = binary_bundle["label_mapping"]
binary_threshold = float(binary_bundle["decision_threshold"])

multiclass_model = multiclass_bundle["model"]
multiclass_imputer = multiclass_bundle["imputer"]
multiclass_fea

In [ ]:

from pathlib import Path

api_file = ROOT / "api" / "main.py"
source = api_file.read_text()

if '@app.post("/predict/batch"' in source:
    print("Batch endpoint already exists. No changes made.")
else:
    batch_endpoint = r'''

class BatchFlowRecord(BaseModel):
    records: list[FlowRecord] = Field(min_length=1, max_length=500)


class BatchPredictionResponse(BaseModel):
    predictions: list[PredictionResponse]
    count: int


@app.post("/predict/batch", response_model=BatchPredictionResponse)
def predict_batch(batch: BatchFlowRecord):
    # Validate all records before inference.
    for index, record in enumerate(batch.records):
        missing = [
            name for name in FEATURES
            if name not in record.features
        ]
        if missing:
            raise HTTPException(
                status_code=422,
                detail={
                    "message": f"Record {index} is missing required features.",
                    "record_index": index,
                    "missing_features": missing,
                },
            )

    try:
        # Construct one feature matrix for the entire batch.
        X = pd.DataFrame(
            [
                [record.features[name] for name in FEATURES]
                for record in batch.records
            ],
            columns=FEATURES,
        )

        X = X.replace([np.inf, -np.inf], np.nan)

        # Stage 1: binary classification.
        X_binary = binary_imputer.transform(X)
        binary_probs = binary_model.predict_proba(X_binary)
        attack_probabilities = binary_probs[:, attack_probability_index]

        # Stage 2: multiclass classification.
        # Run the multiclass model only on records routed as attacks.
        attack_mask = attack_probabilities >= binary_threshold
        predictions = [None] * len(batch.records)

        for i, probability in enumerate(attack_probabilities):
            if not attack_mask[i]:
                predictions[i] = PredictionResponse(
                    binary_decision="BENIGN",
                    attack_probability=float(probability),
                    predicted_label="BENIGN",
                    multiclass_confidence=None,
                    threshold=binary_threshold,
                )

        attack_indices = np.flatnonzero(attack_mask)

        if len(attack_indices) > 0:
            X_attack = X.iloc[attack_indices].to_numpy()
            X_multi = multiclass_imputer.transform(X_attack)
            multiclass_probs = multiclass_model.predict_proba(X_multi)

            for position, original_index in enumerate(attack_indices):
                predicted_id = int(np.argmax(multiclass_probs[position]))

                predictions[original_index] = PredictionResponse(
                    binary_decision="ATTACK",
                    attack_probability=float(
                        attack_probabilities[original_index]
                    ),
                    predicted_label=class_names[predicted_id],
                    multiclass_confidence=float(
                        multiclass_probs[position, predicted_id]
                    ),
                    threshold=binary_threshold,
                )

        return BatchPredictionResponse(
            predictions=predictions,
            count=len(predictions),
        )

    except HTTPException:
        raise
    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=f"Batch inference failed: {type(exc).__name__}: {exc}",
        )
'''

    api_file.write_text(source.rstrip() + "\n\n" + batch_endpoint.lstrip())
    print("Batch endpoint added to:", api_file)


Batch endpoint added to: /content/drive/MyDrive/H-IDS/api/main.py


In [ ]:

import importlib
import main

importlib.invalidate_caches()
main = importlib.reload(main)

from fastapi.testclient import TestClient

client = TestClient(main.app)

print(client.get("/health").status_code)
print("Batch route registered:", any(
    route.path == "/predict/batch"
    for route in main.app.routes
))


200
Batch route registered: True


/usr/lib/python3.13/pickle.py:1754: UserWarning: [04:18:02] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:439: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [04:18:02] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [04:18:02] WARNING: /__w/xgboost/xgboost/src/context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)


In [ ]:

import numpy as np

sample_df = val_df.sample(
    n=min(100, len(val_df)),
    random_state=42
)

records = [
    {
        "features": {
            feature: float(row[feature])
            for feature in main.FEATURES
        }
    }
    for _, row in sample_df.iterrows()
]

response = client.post(
    "/predict/batch",
    json={"records": records}
)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    result = response.json()
    predictions = result["predictions"]

    actual = sample_df["label"].astype(str).tolist()
    predicted = [item["predicted_label"] for item in predictions]

    print("Records submitted:", len(records))
    print("Predictions returned:", result["count"])
    print(
        "All predictions returned:",
        len(predictions) == len(records)
    )
    print(
        "Batch accuracy:",
        round(float(np.mean(np.array(actual) == np.array(predicted))), 6)
    )
    print("\nPrediction distribution:")
    print(pd.Series(predicted).value_counts())
else:
    print(response.text)


HTTP status: 200
Records submitted: 100
Predictions returned: 100
All predictions returned: True
Batch accuracy: 1.0

Prediction distribution:
BENIGN                    74
DoS Hulk                  11
PortScan                   8
DDoS                       5
Web Attack Brute Force     1
DoS slowloris              1
Name: count, dtype: int64


In [ ]:
bad_record = {
    "features": {
        main.FEATURES[0]: 123.0
    }
}

response = client.post(
    "/predict/batch",
    json={"records": [bad_record]}
)

print("HTTP status:", response.status_code)
print(response.json())

HTTP status: 422
{'detail': {'message': 'Record 0 is missing required features.', 'record_index': 0, 'missing_features': ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/H-IDS")
api_file = ROOT / "api" / "main.py"

print("File exists:", api_file.exists())
print("File path:", api_file)
print(api_file.read_text()[:1500])

File exists: True
File path: /content/drive/MyDrive/H-IDS/api/main.py

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field


PROJECT_ROOT = Path(__file__).resolve().parents[1]

BINARY_PATH = (
    PROJECT_ROOT / "models" / "binary" / "xgboost_binary_bundle.joblib"
)
MULTICLASS_PATH = (
    PROJECT_ROOT / "models" / "multiclass" /
    "xgboost_multiclass_bundle.joblib"
)

for path in [BINARY_PATH, MULTICLASS_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"Model bundle not found: {path}")

binary_bundle = joblib.load(BINARY_PATH)
multiclass_bundle = joblib.load(MULTICLASS_PATH)

binary_model = binary_bundle["model"]
binary_imputer = binary_bundle["imputer"]
binary_features = list(binary_bundle["feature_columns"])
binary_mapping = binary_bundle["label_mapping"]
binary_threshold = float(binary_bundle["decision_threshold"])

multiclass_model = multiclass_bundle["mo

In [ ]:

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/H-IDS")
api_file = ROOT / "api" / "main.py"

assert api_file.exists(), f"File not found: {api_file}"

source = api_file.read_text()

# Add the CORS import if it is not already present.
cors_import = "from fastapi.middleware.cors import CORSMiddleware"

if cors_import not in source:
    source = source.replace(
        "from fastapi import FastAPI, HTTPException",
        "from fastapi import FastAPI, HTTPException\n"
        + cors_import,
        1,
    )

# Add the middleware immediately after the FastAPI app declaration.
if "app.add_middleware(" not in source:
    marker = "    version=\"1.0.0\",\n)"
    middleware = '''

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:5173",
        "http://127.0.0.1:5173",
    ],
    allow_credentials=False,
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["Content-Type"],
)
'''
    assert marker in source, (
        "Could not find the expected FastAPI declaration. "
        "No changes have been saved."
    )
    source = source.replace(marker, marker + middleware, 1)

# Save a backup before updating the file.
backup = api_file.with_suffix(".py.bak")
if not backup.exists():
    backup.write_text(api_file.read_text())

api_file.write_text(source)

print("Updated:", api_file)
print("Backup:", backup)
print("CORS import present:", cors_import in source)
print("Middleware present:", "app.add_middleware(" in source)


Updated: /content/drive/MyDrive/H-IDS/api/main.py
Backup: /content/drive/MyDrive/H-IDS/api/main.py.bak
CORS import present: True
Middleware present: True


In [ ]:
import subprocess

result = subprocess.run(
    ["bash", "-lc", "ps -ef | grep '[u]vicorn'"],
    capture_output=True,
    text=True
)

print(result.stdout or "No Uvicorn process found.")

No Uvicorn process found.


In [7]:
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/H-IDS")
API_DIR = ROOT / "api"
LOG_FILE = ROOT / "outputs" / "metrics" / "api_server.log"

LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

api_log = open(LOG_FILE, "w")

api_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "main:app",
        "--app-dir",
        str(API_DIR),
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
    ],
    stdout=api_log,
    stderr=subprocess.STDOUT,
)

time.sleep(8)

if api_process.poll() is not None:
    print(LOG_FILE.read_text(errors="replace"))
    raise RuntimeError("FastAPI did not start successfully.")

print("FastAPI started. PID:", api_process.pid)

FastAPI started. PID: 10502


In [8]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=30,
)

print("HTTP status:", response.status_code)
print("Response:", response.json())

HTTP status: 200
Response: {'status': 'ok', 'binary_model_loaded': True, 'multiclass_model_loaded': True, 'feature_count': 78, 'class_count': 15}


In [11]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb
!dpkg -i /tmp/cloudflared.deb
!cloudflared --version

(Reading database ... 122817 files and directories currently installed.)
Preparing to unpack /tmp/cloudflared.deb ...
Unpacking cloudflared (2026.10.7) over (2026.10.7) ...
Setting up cloudflared (2026.10.7) ...
Processing triggers for man-db (2.12.0-4build2) ...
cloudflared version 2026.10.7 (built 2026-10-30-17:05 UTC)


In [12]:
import subprocess
import time
import re
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/H-IDS")
TUNNEL_LOG = ROOT / "outputs" / "metrics" / "cloudflare_tunnel.log"
TUNNEL_LOG.parent.mkdir(parents=True, exist_ok=True)

tunnel_log_handle = open(TUNNEL_LOG, "w")

tunnel_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8000",
        "--no-autoupdate",
    ],
    stdout=tunnel_log_handle,
    stderr=subprocess.STDOUT,
)

tunnel_url = None

for _ in range(30):
    time.sleep(2)
    log_text = TUNNEL_LOG.read_text(errors="replace")
    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        log_text,
    )
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url is None:
    print(log_text[-5000:])
    raise RuntimeError("Could not obtain the tunnel URL.")

print("API URL:", tunnel_url)
print("Health URL:", tunnel_url + "/health")

API URL: https://levels-atomic-assured-scale.trycloudflare.com
Health URL: https://levels-atomic-assured-scale.trycloudflare.com/health


In [13]:
import requests

response = requests.get(
    tunnel_url + "/health",
    timeout=30,
)

print("HTTP status:", response.status_code)
print("Response:", response.json())

assert response.status_code == 200


HTTP status: 200
Response: {'status': 'ok', 'binary_model_loaded': True, 'multiclass_model_loaded': True, 'feature_count': 78, 'class_count': 15}


In [ ]:
response = requests.get(
    tunnel_url + "/features",
    timeout=30,
)

print("HTTP status:", response.status_code)
print("Feature count:", response.json()["feature_count"])

HTTP status: 200
Feature count: 78


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://127.0.0.1:8000

2026-10-28T04:59:59Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-28T04:59:59Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-28T05:00:03Z INF +--------------------------------------------------------------------------------------------+
2026-10-28T05:00:03Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-28T05:00:03Z INF |  https://drilling-cars-knowledgestorm-planet.trycloudf

In [ ]:
import requests

try:
    response = requests.get(
        "http://127.0.0.1:8000/health",
        timeout=10
    )
    print("HTTP status:", response.status_code)
    print("Response:", response.text)
except Exception as e:
    print("API connection failed:", repr(e))

API connection failed: ConnectionError(MaxRetryError("HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7dbfa1a6c2d0>: Failed to establish a new connection: [Errno 111] Connection refused'))"))


In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/H-IDS")

paths = [
    project / "api" / "main.py",
    project / "models" / "binary" / "xgboost_binary_bundle.joblib",
    project / "models" / "multiclass" / "xgboost_multiclass_bundle.joblib",
]

for path in paths:
    print(f"{'FOUND' if path.exists() else 'MISSING'}: {path}")

FOUND: /content/drive/MyDrive/H-IDS/api/main.py
FOUND: /content/drive/MyDrive/H-IDS/models/binary/xgboost_binary_bundle.joblib
FOUND: /content/drive/MyDrive/H-IDS/models/multiclass/xgboost_multiclass_bundle.joblib


In [ ]:
import subprocess
import sys
import os
import time
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/H-IDS")
log_path = Path("/content/hids_api.log")

log_file = open(log_path, "w")

api_process = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn",
        "api.main:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    cwd=str(project_dir),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

time.sleep(5)

print("FastAPI PID:", api_process.pid)
print("Process running:", api_process.poll() is None)
print("Log file:", log_path)
print(log_path.read_text()[-3000:])

FastAPI PID: 12139
Process running: True
Log file: /content/hids_api.log
/usr/lib/python3.13/pickle.py:1754: UserWarning: [07:22:03] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:439: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [07:22:03] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [07:22:03] WARNING: /__w/xgboost/xgboost/src/context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [07:22:04] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:439: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.13/pickle.py:1754: UserWarning: [07:22:04] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setti

In [ ]:
import requests

try:
    response = requests.get(
        "http://127.0.0.1:8000/health",
        timeout=10
    )
    print("HTTP status:", response.status_code)
    print("Response:", response.text)
except Exception as exc:
    print("Health check failed:", repr(exc))

HTTP status: 200
Response: {"status":"ok","binary_model_loaded":true,"multiclass_model_loaded":true,"feature_count":78,"class_count":15}


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
! /content/cloudflared --version

cloudflared version 2026.10.7 (built 2026-10-30-17:05 UTC)


In [ ]:
import subprocess
import time
from pathlib import Path
import shutil
import re

project_dir = Path("/content/drive/MyDrive/H-IDS")
log_path = Path("/content/hids_tunnel.log")

# Find the existing Cloudflare executable
candidates = [
    shutil.which("cloudflared"),
    str(project_dir / "cloudflared"),
    "/content/cloudflared",
]

cloudflared_path = next(
    (
        Path(p) for p in candidates
        if p and Path(p).is_file()
    ),
    None,
)

if cloudflared_path is None:
    raise FileNotFoundError(
        "cloudflared executable not found. "
        "Tell me this error before proceeding."
    )

log_file = open(log_path, "w")

tunnel_process = subprocess.Popen(
    [
        str(cloudflared_path),
        "tunnel",
        "--url",
        "http://127.0.0.1:8000",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

time.sleep(8)

log_text = log_path.read_text(errors="replace")
print("Tunnel PID:", tunnel_process.pid)
print(log_text[-5000:])

match = re.search(
    r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
    log_text,
)

if match:
    print("\nYOUR NEW TUNNEL URL:", match.group(0))
else:
    print("\nTunnel URL not detected yet. Check the log again.")

Tunnel PID: 12323
2026-11-02T07:22:41Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-02T07:22:41Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-02T07:22:45Z INF +--------------------------------------------------------------------------------------------+
2026-11-02T07:22:45Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-02T07:22:45Z INF |  https://laura-measuring-spot-people

In [ ]:
from pathlib import Path
import requests
import re

# Check the local FastAPI server
try:
    r = requests.get(
        "http://127.0.0.1:8000/health",
        timeout=5
    )
    print("LOCAL API:", r.status_code, r.text)
except Exception as e:
    print("LOCAL API ERROR:", repr(e))

# Find the latest URL in the tunnel log
log_path = Path("/content/hids_tunnel.log")

if log_path.exists():
    log = log_path.read_text(errors="replace")
    urls = re.findall(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        log
    )
    print("TUNNEL URL:", urls[-1] if urls else "No URL found")
    print("TUNNEL LOG (last 1500 characters):")
    print(log[-1500:])
else:
    print("Tunnel log not found.")

LOCAL API: 200 {"status":"ok","binary_model_loaded":true,"multiclass_model_loaded":true,"feature_count":78,"class_count":15}
TUNNEL URL: https://laura-measuring-spot-people.trycloudflare.com
TUNNEL LOG (last 1500 characters):
-----------------------------------------------------------------+
2026-11-02T07:22:51Z INF precheck component="DNS Resolution" details="DNS Resolved successfully" run_id=75f5eefb-feb9-4d66-8dc7-15eedd6e9d3a status=pass target=region1.v2.argotunnel.com
2026-11-02T07:22:51Z INF precheck component="DNS Resolution" details="DNS Resolved successfully" run_id=75f5eefb-feb9-4d66-8dc7-15eedd6e9d3a status=pass target=region2.v2.argotunnel.com
2026-11-02T07:22:51Z INF precheck component="UDP Connectivity" details="QUIC connection successful" run_id=75f5eefb-feb9-4d66-8dc7-15eedd6e9d3a status=pass target=region1.v2.argotunnel.com
2026-11-02T07:22:51Z INF precheck component="UDP Connectivity" details="QUIC connection successful" run_id=75f5eefb-feb9-4d66-8dc7-15eedd6e9d3a st

In [ ]:

import requests
import pandas as pd
from google.colab import files

# Get the exact feature names expected by your API
response = requests.get(
    "http://127.0.0.1:8000/features",
    timeout=10
)
response.raise_for_status()

features = response.json()["features"]
assert len(features) == 78, f"Expected 78 features, got {len(features)}"

# Create 10 synthetic numeric records for integration testing
df = pd.DataFrame(0.0, index=range(10), columns=features)

# Save and download the CSV
output_file = "/content/hids_test_10_flows.csv"
df.to_csv(output_file, index=False)

print("CSV created:", output_file)
print("Rows:", len(df))
print("Feature columns:", len(df.columns))
print(df.head())

files.download(output_file)


CSV created: /content/hids_test_10_flows.csv
Rows: 10
Feature columns: 78
   Destination Port  Flow Duration  Total Fwd Packets  Total Backward Packets  \
0               0.0            0.0                0.0                     0.0   
1               0.0            0.0                0.0                     0.0   
2               0.0            0.0                0.0                     0.0   
3               0.0            0.0                0.0                     0.0   
4               0.0            0.0                0.0                     0.0   

   Total Length of Fwd Packets  Total Length of Bwd Packets  \
0                          0.0                          0.0   
1                          0.0                          0.0   
2                          0.0                          0.0   
3                          0.0                          0.0   
4                          0.0                          0.0   

   Fwd Packet Length Max  Fwd Packet Length Min  Fwd Packet 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/H-IDS')

required_files = [
    'api/main.py',
    'models/binary/xgboost_binary_bundle.joblib',
    'models/multiclass/xgboost_multiclass_bundle.joblib'
]

for file in required_files:
    path = PROJECT_DIR / file
    print(f"{'FOUND' if path.exists() else 'MISSING'}: {file}")

FOUND: api/main.py
FOUND: models/binary/xgboost_binary_bundle.joblib
FOUND: models/multiclass/xgboost_multiclass_bundle.joblib


In [ ]:

import subprocess
import sys
import time
import socket
import requests
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/H-IDS")
LOG_PATH = Path("/content/hids_api.log")

# Check whether port 8000 is already responding
sock = socket.socket()
port_open = sock.connect_ex(("127.0.0.1", 8000)) == 0
sock.close()

if port_open:
    try:
        response = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=5
        )
        print("Existing API:", response.status_code)
        print(response.text)
    except Exception:
        print("Port 8000 is occupied, but health check failed.")
else:
    log_file = open(LOG_PATH, "w")

    api_process = subprocess.Popen(
        [
            sys.executable,
            "-m", "uvicorn",
            "api.main:app",
            "--host", "0.0.0.0",
            "--port", "8000"
        ],
        cwd=str(PROJECT_DIR),
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

    print("FastAPI process started. PID:", api_process.pid)

    # Wait for API readiness
    for attempt in range(15):
        time.sleep(2)
        try:
            response = requests.get(
                "http://127.0.0.1:8000/health",
                timeout=5
            )
            if response.status_code == 200:
                print("FastAPI is ready!")
                print(response.json())
                break
        except requests.RequestException:
            print(f"Waiting for API... {attempt + 1}/15")
    else:
        print("API did not become ready. Check the log:")
        print(LOG_PATH.read_text(errors="replace")[-4000:])


FastAPI process started. PID: 3088
Waiting for API... 1/15
Waiting for API... 2/15
FastAPI is ready!
{'status': 'ok', 'binary_model_loaded': True, 'multiclass_model_loaded': True, 'feature_count': 78, 'class_count': 15}


In [ ]:

import subprocess
import time
import re
from pathlib import Path

CLOUDFLARED = Path("/content/cloudflared")
TUNNEL_LOG = Path("/content/hids_tunnel.log")

# Download cloudflared if it is missing
if not CLOUDFLARED.exists():
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", str(CLOUDFLARED)
    ], check=True)

CLOUDFLARED.chmod(0o755)

# Start tunnel in background
log_file = open(TUNNEL_LOG, "w")

tunnel_process = subprocess.Popen(
    [
        str(CLOUDFLARED),
        "tunnel",
        "--url",
        "http://127.0.0.1:8000"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("Cloudflare tunnel process PID:", tunnel_process.pid)

# Wait for the public URL
for attempt in range(20):
    time.sleep(2)
    log = TUNNEL_LOG.read_text(errors="replace")
    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        log
    )

    if match:
        tunnel_url = match.group(0)
        print("\nNEW TUNNEL URL:", tunnel_url)
        break
else:
    print("Tunnel URL not found. Recent logs:")
    print(log[-3000:])


Cloudflare tunnel process PID: 3146

NEW TUNNEL URL: https://foreign-jackson-wants-lloyd.trycloudflare.com


In [ ]:
import requests

TUNNEL_URL = "https://foreign-jackson-wants-lloyd.trycloudflare.com"

response = requests.get(
    f"{TUNNEL_URL}/health",
    timeout=20
)

print("HTTP:", response.status_code)
print("Response:", response.json())

HTTP: 200
Response: {'status': 'ok', 'binary_model_loaded': True, 'multiclass_model_loaded': True, 'feature_count': 78, 'class_count': 15}


In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/H-IDS/data")

print("CSV files available:\n")

for file in DATA_DIR.rglob("*.csv"):
    print(file.name, "|", round(file.stat().st_size / (1024**2), 2), "MB")

CSV files available:

Wednesday-workingHours.pcap_ISCX.csv | 214.74 MB
Tuesday-WorkingHours.pcap_ISCX.csv | 128.82 MB
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv | 49.61 MB
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv | 79.25 MB
Monday-WorkingHours.pcap_ISCX.csv | 168.73 MB
Friday-WorkingHours-Morning.pcap_ISCX.csv | 55.62 MB
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv | 73.34 MB
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv | 73.55 MB


In [ ]:

import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/H-IDS/data")

files_to_check = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
]

for filename in files_to_check:
    file_path = DATA_DIR / filename

    print("\n" + "=" * 70)
    print("FILE:", filename)

    df = pd.read_csv(file_path, nrows=5)
    df.columns = df.columns.str.strip()

    print("\nNumber of columns:", len(df.columns))
    print("\nColumn names:")
    print(df.columns.tolist())

    label_columns = [
        col for col in df.columns
        if col.lower() in ["label", "label_raw"]
    ]

    if label_columns:
        print("\nSample labels:")
        print(df[label_columns].head())
    else:
        print("\nNo label column detected in first five rows.")



FILE: Monday-WorkingHours.pcap_ISCX.csv


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/H-IDS/data/Monday-WorkingHours.pcap_ISCX.csv'

In [ ]:

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/H-IDS")

print("Searching for CIC-IDS2017 CSV files...\n")

for file in PROJECT_DIR.rglob("*.csv"):
    if "Monday-WorkingHours" in file.name or "Friday-WorkingHours-Afternoon-DDoS" in file.name:
        print("FOUND:", file)


Searching for CIC-IDS2017 CSV files...

FOUND: /content/drive/MyDrive/H-IDS/data/raw/Monday-WorkingHours.pcap_ISCX.csv


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/H-IDS")

target_files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDoS.pcap_ISCX.csv"
]

for filename in target_files:
    matches = list(PROJECT_DIR.rglob(filename))

    if not matches:
        print(f"MISSING: {filename}")
        continue

    file_path = matches[0]

    print("\n" + "=" * 70)
    print("FILE:", file_path)

    df = pd.read_csv(file_path, nrows=5)
    df.columns = df.columns.str.strip()

    print("Number of columns:", len(df.columns))
    print("Column names:")
    print(df.columns.tolist())

    if "Label" in df.columns:
        print("\nLabels:")
        print(df["Label"].value_counts())



FILE: /content/drive/MyDrive/H-IDS/data/raw/Monday-WorkingHours.pcap_ISCX.csv
Number of columns: 79
Column names:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Fla

In [ ]:

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/H-IDS")

print("Friday CSV files found:\n")

for file in PROJECT_DIR.rglob("*.csv"):
    if "friday" in file.name.lower():
        print(file)


Friday CSV files found:

/content/drive/MyDrive/H-IDS/data/raw/Friday-WorkingHours-Morning.pcap_ISCX.csv
/content/drive/MyDrive/H-IDS/data/raw/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
/content/drive/MyDrive/H-IDS/data/raw/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/H-IDS")

monday_path = PROJECT_DIR / "data/raw/Monday-WorkingHours.pcap_ISCX.csv"

friday_matches = [
    p for p in PROJECT_DIR.rglob("*.csv")
    if "friday-workinghours-afternoon" in p.name.lower()
    and "ddos" in p.name.lower()
]

if not friday_matches:
    raise FileNotFoundError("Friday DDoS CSV not found.")

friday_path = friday_matches[0]

for name, path in [
    ("BENIGN DATASET", monday_path),
    ("DDOS DATASET", friday_path)
]:
    print("\n", "=" * 60)
    print(name)
    print("File:", path)

    df = pd.read_csv(path, nrows=1000)
    df.columns = df.columns.str.strip()

    print("Columns:", len(df.columns))
    print("Label distribution in first 1000 rows:")
    print(df["Label"].value_counts(dropna=False))



BENIGN DATASET
File: /content/drive/MyDrive/H-IDS/data/raw/Monday-WorkingHours.pcap_ISCX.csv
Columns: 79
Label distribution in first 1000 rows:
Label
BENIGN    1000
Name: count, dtype: int64

DDOS DATASET
File: /content/drive/MyDrive/H-IDS/data/raw/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Columns: 79
Label distribution in first 1000 rows:
Label
BENIGN    1000
Name: count, dtype: int64


In [ ]:

import pandas as pd
import numpy as np
import requests
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/H-IDS")

MONDAY_FILE = (
    PROJECT_DIR / "data/raw/Monday-WorkingHours.pcap_ISCX.csv"
)

FRIDAY_FILE = (
    PROJECT_DIR / "data/raw/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
)

TUNNEL_URL = "https://genetics-across-photograph-toddler.trycloudflare.com"

# 1. Get the exact feature order expected by FastAPI
response = requests.get(f"{TUNNEL_URL}/features", timeout=30)
response.raise_for_status()

api_features = response.json()["features"]

print("API feature count:", len(api_features))

if len(api_features) != 78:
    raise ValueError("Expected 78 API features.")

# 2. Find one valid BENIGN record from Monday
benign_record = None

for chunk in pd.read_csv(MONDAY_FILE, chunksize=50000):
    chunk.columns = chunk.columns.str.strip()

    candidates = chunk[
        chunk["Label"].astype(str).str.strip().str.upper() == "BENIGN"
    ]

    for _, row in candidates.iterrows():
        values = pd.to_numeric(row[api_features], errors="coerce")

        if np.isfinite(values.to_numpy(dtype=float)).all():
            benign_record = values
            break

    if benign_record is not None:
        break

if benign_record is None:
    raise ValueError("Could not find a valid BENIGN record.")

print("Found a valid BENIGN record.")

# 3. Search the entire Friday file for a genuine DDoS record
ddos_record = None
ddos_row_number = None
rows_scanned = 0

for chunk in pd.read_csv(FRIDAY_FILE, chunksize=50000):
    chunk.columns = chunk.columns.str.strip()

    candidates = chunk[
        chunk["Label"].astype(str).str.strip().str.upper() == "DDOS"
    ]

    for index, row in candidates.iterrows():
        values = pd.to_numeric(row[api_features], errors="coerce")

        if np.isfinite(values.to_numpy(dtype=float)).all():
            ddos_record = values
            ddos_row_number = rows_scanned + index
            break

    if ddos_record is not None:
        break

    rows_scanned += len(chunk)

if ddos_record is None:
    raise ValueError(
        "No valid DDoS record found. Check the Friday file's label values."
    )

print("Found a valid DDoS record.")
print("DDoS row position:", ddos_row_number)

# 4. Create a two-record CSV in the exact API feature order
test_df = pd.DataFrame(
    [benign_record.to_dict(), ddos_record.to_dict()],
    columns=api_features
)

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_file = OUTPUT_DIR / "real_flow_test.csv"
test_df.to_csv(output_file, index=False)

print("\nTest CSV created:", output_file)
print("Shape:", test_df.shape)
print("Expected labels: BENIGN, DDoS")
print("Feature order matches API:", list(test_df.columns) == api_features)


API feature count: 78
Found a valid BENIGN record.
Found a valid DDoS record.
DDoS row position: 18883

Test CSV created: /content/drive/MyDrive/H-IDS/outputs/real_flow_test.csv
Shape: (2, 78)
Expected labels: BENIGN, DDoS
Feature order matches API: True


In [ ]:
print(test_df.shape)
print(test_df.head(2))
print("All values finite:", np.isfinite(test_df.to_numpy()).all())

(2, 78)
   Destination Port  Flow Duration  Total Fwd Packets  Total Backward Packets  \
0           49188.0            4.0                2.0                     0.0   
1              80.0      1293792.0                3.0                     7.0   

   Total Length of Fwd Packets  Total Length of Bwd Packets  \
0                         12.0                          0.0   
1                         26.0                      11607.0   

   Fwd Packet Length Max  Fwd Packet Length Min  Fwd Packet Length Mean  \
0                    6.0                    6.0                6.000000   
1                   20.0                    0.0                8.666667   

   Fwd Packet Length Std  ...  act_data_pkt_fwd  min_seg_size_forward  \
0               0.000000  ...               1.0                  20.0   
1              10.263203  ...               2.0                  20.0   

   Active Mean  Active Std  Active Max  Active Min  Idle Mean  Idle Std  \
0          0.0         0.0         0.

In [ ]:

import pandas as pd
import numpy as np
import requests
from pathlib import Path
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/H-IDS")

MONDAY_FILE = (
    PROJECT_DIR / "data/raw/Monday-WorkingHours.pcap_ISCX.csv"
)

FRIDAY_FILE = (
    PROJECT_DIR / "data/raw/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
)

API_URL = "https://genetics-across-photograph-toddler.trycloudflare.com"

SAMPLE_SIZE = 1000
BATCH_SIZE = 100

# --------------------------------------------------
# 2. CHECK API AND LOAD FEATURE ORDER
# --------------------------------------------------

health = requests.get(f"{API_URL}/health", timeout=30)
health.raise_for_status()

print("API health:", health.json())

feature_response = requests.get(
    f"{API_URL}/features", timeout=30
)
feature_response.raise_for_status()

API_FEATURES = feature_response.json()["features"]

assert len(API_FEATURES) == 78, (
    f"Expected 78 features, received {len(API_FEATURES)}"
)

print("API features loaded:", len(API_FEATURES))

# --------------------------------------------------
# 3. EXTRACT VALID RECORDS BY GROUND-TRUTH LABEL
# --------------------------------------------------

def collect_records(file_path, target_label, sample_size):
    collected = []
    total_collected = 0

    for chunk in pd.read_csv(file_path, chunksize=50000):
        chunk.columns = chunk.columns.str.strip()

        labels = chunk["Label"].astype(str).str.strip().str.upper()
        mask = labels == target_label.upper()

        candidates = chunk.loc[mask, API_FEATURES]

        if candidates.empty:
            continue

        numeric = candidates.apply(pd.to_numeric, errors="coerce")
        numeric = numeric.replace([np.inf, -np.inf], np.nan)
        numeric = numeric.dropna()

        remaining = sample_size - total_collected

        if remaining <= 0:
            break

        selected = numeric.head(remaining)

        if not selected.empty:
            collected.append(selected)
            total_collected += len(selected)

        if total_collected >= sample_size:
            break

    if total_collected < sample_size:
        raise ValueError(
            f"Only found {total_collected} valid {target_label} records "
            f"in {file_path.name}; needed {sample_size}."
        )

    return pd.concat(collected, ignore_index=True)


print("\nExtracting BENIGN records...")
benign_df = collect_records(
    MONDAY_FILE, "BENIGN", SAMPLE_SIZE
)

print("Extracting DDoS records...")
ddos_df = collect_records(
    FRIDAY_FILE, "DDoS", SAMPLE_SIZE
)

# Combine records and preserve ground-truth labels
test_df = pd.concat(
    [benign_df, ddos_df],
    ignore_index=True
)

y_true_multiclass = (
    ["BENIGN"] * len(benign_df)
    + ["DDoS"] * len(ddos_df)
)

y_true_binary = (
    ["BENIGN"] * len(benign_df)
    + ["ATTACK"] * len(ddos_df)
)

print("\nDataset prepared:", test_df.shape)
print("Ground-truth distribution:")
print(pd.Series(y_true_multiclass).value_counts())

assert list(test_df.columns) == API_FEATURES
assert np.isfinite(test_df.to_numpy(dtype=float)).all()

# --------------------------------------------------
# 4. SEND RECORDS TO FASTAPI IN BATCHES
# --------------------------------------------------

all_predictions = []

for start in range(0, len(test_df), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(test_df))

    batch_df = test_df.iloc[start:end]

    records = [
        {"features": row}
        for row in batch_df.to_dict(orient="records")
    ]

    response = requests.post(
        f"{API_URL}/predict/batch",
        json={"records": records},
        timeout=120
    )

    response.raise_for_status()

    batch_result = response.json()
    batch_predictions = batch_result["predictions"]

    if len(batch_predictions) != len(records):
        raise ValueError(
            f"Batch {start}:{end} returned an unexpected prediction count."
        )

    all_predictions.extend(batch_predictions)

    print(
        f"Processed {end}/{len(test_df)} records"
    )

assert len(all_predictions) == len(test_df)

print("\nAll predictions received successfully.")

# --------------------------------------------------
# 5. EXTRACT PREDICTED LABELS
# --------------------------------------------------

def normalize_label(label):
    label = str(label).strip().upper()

    if label == "DDOS":
        return "DDoS"

    return label


y_pred_binary = [
    str(p["binary_decision"]).strip().upper()
    for p in all_predictions
]

y_pred_multiclass = [
    normalize_label(p["predicted_label"])
    for p in all_predictions
]

# --------------------------------------------------
# 6. BINARY CLASSIFICATION METRICS
# --------------------------------------------------

print("\n" + "=" * 60)
print("BINARY IDS EVALUATION")
print("=" * 60)

print("Accuracy:", accuracy_score(y_true_binary, y_pred_binary))
print(
    "Precision:",
    precision_score(
        y_true_binary, y_pred_binary,
        pos_label="ATTACK", zero_division=0
    )
)
print(
    "Recall:",
    recall_score(
        y_true_binary, y_pred_binary,
        pos_label="ATTACK", zero_division=0
    )
)
print(
    "F1-score:",
    f1_score(
        y_true_binary, y_pred_binary,
        pos_label="ATTACK", zero_division=0
    )
)

print("\nBinary confusion matrix:")
print(
    confusion_matrix(
        y_true_binary,
        y_pred_binary,
        labels=["BENIGN", "ATTACK"]
    )
)

print("\nBinary classification report:")
print(
    classification_report(
        y_true_binary,
        y_pred_binary,
        labels=["BENIGN", "ATTACK"],
        zero_division=0
    )
)

# --------------------------------------------------
# 7. MULTICLASS CLASSIFICATION METRICS
# --------------------------------------------------

print("\n" + "=" * 60)
print("MULTICLASS IDS EVALUATION")
print("=" * 60)

print("Accuracy:", accuracy_score(
    y_true_multiclass, y_pred_multiclass
))

print("\nMulticlass classification report:")
print(
    classification_report(
        y_true_multiclass,
        y_pred_multiclass,
        labels=["BENIGN", "DDoS"],
        zero_division=0
    )
)

print("\nMulticlass confusion matrix:")
print(
    confusion_matrix(
        y_true_multiclass,
        y_pred_multiclass,
        labels=["BENIGN", "DDoS"]
    )
)

# --------------------------------------------------
# 8. SAVE DETAILED RESULTS
# --------------------------------------------------

results_df = pd.DataFrame({
    "ground_truth": y_true_multiclass,
    "binary_decision": y_pred_binary,
    "predicted_label": y_pred_multiclass,
    "attack_probability": [
        p["attack_probability"] for p in all_predictions
    ],
    "multiclass_confidence": [
        p.get("multiclass_confidence") for p in all_predictions
    ]
})

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_path = OUTPUT_DIR / "batch_validation_2000.csv"
results_df.to_csv(results_path, index=False)

print("\nResults saved to:", results_path)
print("Total evaluated:", len(results_df))


API health: {'status': 'ok', 'binary_model_loaded': True, 'multiclass_model_loaded': True, 'feature_count': 78, 'class_count': 15}
API features loaded: 78

Extracting BENIGN records...
Extracting DDoS records...

Dataset prepared: (2000, 78)
Ground-truth distribution:
BENIGN    1000
DDoS      1000
Name: count, dtype: int64
Processed 100/2000 records
Processed 200/2000 records
Processed 300/2000 records
Processed 400/2000 records
Processed 500/2000 records
Processed 600/2000 records
Processed 700/2000 records
Processed 800/2000 records
Processed 900/2000 records
Processed 1000/2000 records
Processed 1100/2000 records
Processed 1200/2000 records
Processed 1300/2000 records
Processed 1400/2000 records
Processed 1500/2000 records
Processed 1600/2000 records
Processed 1700/2000 records
Processed 1800/2000 records
Processed 1900/2000 records
Processed 2000/2000 records

All predictions received successfully.

BINARY IDS EVALUATION
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0

Binar

In [ ]:

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/H-IDS")

print("NOTEBOOKS IN PROJECT:\n")

for file in PROJECT_DIR.rglob("*.ipynb"):
    print(file)


NOTEBOOKS IN PROJECT:

/content/drive/MyDrive/H-IDS/notebooks/00_environment_setup.ipynb
/content/drive/MyDrive/H-IDS/notebooks/01_binary_detector.ipynb
/content/drive/MyDrive/H-IDS/notebooks/03_hybrid_inference.ipynb
/content/drive/MyDrive/H-IDS/notebooks/02_multiclass_detector.ipynb


In [ ]:

import json
from pathlib import Path

NOTEBOOK_DIR = Path(
    "/content/drive/MyDrive/H-IDS/notebooks"
)

notebooks = [
    "01_binary_detector.ipynb",
    "02_multiclass_detector.ipynb"
]

keywords = [
    "train_test_split",
    "train_test",
    "test_size",
    "random_state",
    "stratify",
    "drop_duplicates",
    "read_csv",
    "fit(",
    "fit_transform",
    "save",
    "joblib.dump",
    "X_train",
    "X_test",
    "y_train",
    "y_test"
]

for notebook_name in notebooks:
    notebook_path = NOTEBOOK_DIR / notebook_name

    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    print("\n" + "=" * 90)
    print("NOTEBOOK:", notebook_name)
    print("=" * 90)

    for index, cell in enumerate(notebook["cells"]):
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell["source"])
        matched = [
            keyword for keyword in keywords
            if keyword.lower() in source.lower()
        ]

        if matched:
            print(f"\n--- CELL {index} ---")
            print("Matched:", ", ".join(matched))
            print(source[:6000])

            if len(source) > 6000:
                print("\n[Cell truncated]")



NOTEBOOK: 01_binary_detector.ipynb

--- CELL 4 ---
Matched: X_train, y_train

X_train, y_train = load_binary_split(TRAIN_PATH)
X_val, y_val = load_binary_split(VALIDATION_PATH)

print("TRAIN")
print("X:", X_train.shape, X_train.dtypes.unique())
print("y:", y_train.shape)
print(y_train.value_counts().rename(index={0: "BENIGN", 1: "ATTACK"}))

print("\nVALIDATION")
print("X:", X_val.shape, X_val.dtypes.unique())
print("y:", y_val.shape)
print(y_val.value_counts().rename(index={0: "BENIGN", 1: "ATTACK"}))

assert X_train.shape[1] == 78
assert X_val.shape[1] == 78
assert list(X_train.columns) == list(X_val.columns)
assert not y_train.isna().any()
assert not y_val.isna().any()

print("\nFeature and target checks passed.")


--- CELL 5 ---
Matched: fit_transform, X_train

from sklearn.impute import SimpleImputer

# Fit only on training data to prevent data leakage.
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(